In [1]:
import pandas as pd

In [2]:
from google.cloud import bigquery

In [ ]:
# gcloud auth login
# gcloud auth application-default login

## set project
# gcloud config set project aeo-supplychain-datamart-prod
# gcloud auth application-default set-quota-project aeo-supplychain-datamart-prod

In [3]:
bq_client = bigquery.Client()

In [ ]:
for p in bq_client.list_projects():
    print(p.project_id )
    for d in bq_client.list_datasets(project=p.project_id):
        print(d.dataset_id)
        # for t in bq_client.list_tables(f"{p.project_id}.{d.dataset_id}"):
            # print(t.table_id )

In [5]:
output_filename = 'bigquery_inventory_with_columns.csv'

In [11]:
# 1. Create a list to hold all our row data
all_rows_list = []

In [12]:
# Define the header, which we will use for the DataFrame columns
header = ['project_id', 'dataset_id', 'table_id' , 'field_name' , 'field_type']

In [13]:
for p in bq_client.list_projects():
    project_id = p.project_id
    found_dataset_in_project = False

    for d in bq_client.list_datasets(project=project_id):
        found_dataset_in_project = True
        dataset_id = d.dataset_id
                
        found_table_in_dataset = False
        
        for t in bq_client.list_tables(f"{project_id}.{dataset_id}"):
            found_table_in_dataset = True
            table_id = t.table_id

            full_table = bq_client.get_table(t)

            if not full_table.schema:
                                # Handle tables with no schema (e.g., failed external table)
                all_rows_list.append([project_id, dataset_id, table_id, "N/A (No Schema)", "N/A"])
            else:
                # Loop through each column (SchemaField) in the schema
                for field in full_table.schema:
                    all_rows_list.append([
                        project_id, 
                        dataset_id, 
                        table_id, 
                        field.name, 
                        field.field_type
                    ])
                        
            # 2. Instead of writing to file, append data to our list
            # all_rows_list.append([project_id, dataset_id, table_id])
                    
        if not found_table_in_dataset:
            all_rows_list.append([project_id, dataset_id, "N/A (No Tables)"])
                        
                
    if not found_dataset_in_project:
        # all_rows_list.append([project_id, "N/A (No Datasets)", "N/A"])
        all_rows_list.append([project_id, "N/A (No Datasets)", "N/A", "N/A", "N/A"])

In [14]:
len(all_rows_list)

928745

In [15]:
df = pd.DataFrame(all_rows_list, columns=header)

In [16]:
df.head()

,project_id,dataset_id,table_id,field_name,field_type
0,aeo-gold-stores-prod,N/A (No Datasets),N/A,N/A,N/A
1,aeo-gold-stores-test,N/A (No Datasets),N/A,N/A,N/A
2,aeo-datagold-terraform-e1bf,N/A (No Datasets),N/A,N/A,N/A
3,aeo-identity-terraform-9090,N/A (No Datasets),N/A,N/A,N/A
4,sys-38386478785055756932340087,N/A (No Datasets),N/A,N/A,N/A


In [17]:
df = pd.DataFrame(all_rows_list, columns=header)

In [18]:
df.to_csv(output_filename, index=False)